In [1]:
# Install & Import

import json
import re
import random
from collections import Counter

In [2]:
# Import NCBI MACCROBAT2018 Dataset
import os

def load_macrobot2018(folder_path):
    """Load .ann and .txt files from MACCROBAT2018 dataset"""
    files = sorted([f.replace('.ann', '') for f in os.listdir(folder_path) if f.endswith('.ann')])
    
    data = []
    for file_id in files:
        txt_path = os.path.join(folder_path, f"{file_id}.txt")
        ann_path = os.path.join(folder_path, f"{file_id}.ann")
        
        # Read text
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        # Read annotations
        entities = []
        with open(ann_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('T'):  # Entity annotation
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        info = parts[1].split()
                        label = info[0]
                        # Handle multiple spans (e.g., "1373;1386") - take first span
                        start = int(info[1].split(';')[0])
                        end = int(info[2].split(';')[0])
                        entities.append({
                            "start": start,
                            "end": end,
                            "label": label
                        })
        
        data.append({"text": text, "entities": entities})
    
    return data

# Load the dataset
data_path = r"C:\Users\sanan\OneDrive\Documents\NLP\NLP Capstone Project\MACCROBAT2018"
dataset = load_macrobot2018(data_path)
print(f"Loaded {len(dataset)} documents")
print(f"Sample: {dataset[0]}")

Loaded 200 documents
Sample: {'text': "CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.\nThe symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.\nExcept for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.\nAn electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.\nTransthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).\nThe anterior tricuspid valve leaflet was elongated (Fig.2C, arrow), whereas the septal

In [3]:
# Explore the loaded dataset
print(f"Total documents: {len(dataset)}")

# Show entity types
entity_types = []
for doc in dataset:
    for ent in doc["entities"]:
        entity_types.append(ent["label"])

from collections import Counter
entity_counts = Counter(entity_types)
print(f"\nEntity type distribution:")
for label, count in entity_counts.most_common():
    print(f"  {label}: {count}")

# Show sample document
print(f"\nSample document:")
print(f"Text length: {len(dataset[0]['text'])}")
print(f"Entities: {len(dataset[0]['entities'])}")
print(f"Text preview: {dataset[0]['text'][:200]}...")

Total documents: 200

Entity type distribution:
  Diagnostic_procedure: 4567
  Sign_symptom: 3359
  Biological_structure: 2931
  Detailed_description: 2901
  Lab_value: 2858
  Disease_disorder: 1362
  Medication: 1076
  Therapeutic_procedure: 1005
  Date: 731
  Clinical_event: 626
  History: 392
  Severity: 369
  Dosage: 362
  Nonbiological_location: 354
  Coreference: 313
  Duration: 280
  Age: 206
  Sex: 191
  Administration: 175
  Distance: 122
  Activity: 108
  Family_history: 81
  Frequency: 76
  Shape: 65
  Personal_background: 57
  Time: 57
  Subject: 54
  Color: 52
  Texture: 46
  Area: 43
  Outcome: 42
  Qualitative_concept: 41
  Volume: 33
  Quantitative_concept: 31
  Other_event: 22
  Other_entity: 20
  Occupation: 13
  Biological_attribute: 10
  Height: 4
  Weight: 4
  Mass: 2

Sample document:
Text length: 1686
Entities: 68
Text preview: CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.
The symptoms occurred during rest, 2–3 times 

Tokenization + BIO Conversion

In [4]:
def tokenize_with_offsets(text):
    return [(m.group(), m.start(), m.end())
            for m in re.finditer(r'\w+|\S', text)]


def convert_to_bio(text, entities):
    tokens_with_offsets = tokenize_with_offsets(text)
    tags = ["O"] * len(tokens_with_offsets)

    for ent in entities:
        ent_start, ent_end, label = ent["start"], ent["end"], ent["label"]

        entity_token_indices = []
        for i, (token_text, token_start, token_end) in enumerate(tokens_with_offsets):
            # Check for overlap: token_start < ent_end AND token_end > ent_start
            # This condition means the token's span intersects with the entity's span.
            if token_start < ent_end and token_end > ent_start:
                entity_token_indices.append(i)

        if not entity_token_indices:
            continue

        # Mark the first token of the entity as B-label
        tags[entity_token_indices[0]] = f"B-{label}"

        # Mark subsequent tokens of the entity as I-label
        for i in range(1, len(entity_token_indices)):
            tags[entity_token_indices[i]] = f"I-{label}"

    return [(tokens_with_offsets[i][0], tags[i]) for i in range(len(tokens_with_offsets))]


Load & Convert Dataset


In [5]:
def load_and_convert(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    sentences = []
    for item in data:
        bio = convert_to_bio(item["text"], item["entities"])
        sentences.append(bio)

    return sentences


def save_bio(sentences, filename):
    with open(filename, "w") as f:
        for sent in sentences:
            for token, tag in sent:
                f.write(f"{token} {tag}\n")
            f.write("\n")

Split Dataset

In [6]:
def split_data(sentences, train=0.7, val=0.1, test=0.2, seed=42):
    random.seed(seed)
    random.shuffle(sentences)

    n = len(sentences)
    t = int(n * train)
    v = int(n * val)

    return (
        sentences[:t],
        sentences[t:t+v],
        sentences[t+v:]
    )

Load BIO into Tokens

In [7]:
def load_bio(filepath):
    sentences, tags = [], []
    s, t = [], []

    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                if s:
                    sentences.append(s)
                    tags.append(t)
                    s, t = [], []
            else:
                tok, tag = line.split()
                s.append(tok)
                t.append(tag)

    if s:
        sentences.append(s)
        tags.append(t)

    return sentences, tags

Preprocessing

In [8]:
def clean_token(token):
    token = token.lower()
    token = re.sub(r'\d', '0', token)
    return token


def preprocess(sentences):
    return [[clean_token(w) for w in sent] for sent in sentences]

Build Vocabulary

In [9]:
def build_vocab(sentences, tags):
    word_counter = Counter(w for s in sentences for w in s)
    tag_set = set(t for seq in tags for t in seq)

    word2idx = {"<PAD>": 0, "<UNK>": 1}
    for w in word_counter:
        word2idx[w] = len(word2idx)

    tag2idx = {"<PAD>": 0}
    for t in sorted(tag_set):
        tag2idx[t] = len(tag2idx)

    char2idx = {"<PAD>": 0, "<UNK>": 1}
    for w in word_counter:
        for ch in w:
            if ch not in char2idx:
                char2idx[ch] = len(char2idx)

    return word2idx, tag2idx, char2idx

Encoding

In [10]:
def encode(sentences, tags, word2idx, tag2idx, char2idx):
    X, y, X_char = [], [], []

    for sent, tag_seq in zip(sentences, tags):
        word_ids = [word2idx.get(w, 1) for w in sent]
        tag_ids = [tag2idx[t] for t in tag_seq]

        char_ids = []
        for w in sent:
            char_ids.append([char2idx.get(c, 1) for c in w])

        X.append(word_ids)
        y.append(tag_ids)
        X_char.append(char_ids)

    return X, y, X_char

Padding

In [11]:
def pad(seq, max_len, pad_val=0):
    return [s[:max_len] + [pad_val]*(max_len-len(s)) for s in seq]


def pad_chars(seq, max_len, max_word_len):
    out = []
    for sent in seq:
        s = []
        for w in sent:
            w = w[:max_word_len] + [0]*(max_word_len-len(w))
            s.append(w)
        while len(s) < max_len:
            s.append([0]*max_word_len)
        out.append(s[:max_len])
    return out

Mask

In [12]:
def create_mask(X):
    return [[1 if tok != 0 else 0 for tok in seq] for seq in X]

Run Full Pipeline

In [13]:
import numpy as np
from collections import defaultdict

# ─────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────

def convert_to_bio(text, entities):
    tokens = []
    i = 0
    entities = sorted(entities, key=lambda e: e["start"])
    char_labels = ["O"] * len(text)
    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]
        char_labels[start] = f"B-{label}"
        for j in range(start + 1, end):
            char_labels[j] = f"I-{label}"
    for word in text.split():
        start_idx = text.index(word, i)
        i = start_idx + len(word)
        tag = char_labels[start_idx]
        tokens.append((word, tag))
    return tokens


def save_bio(sentences, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for sent in sentences:
            for token, tag in sent:
                f.write(f"{token} {tag}\n")
            f.write("\n")


def split_data(data, train_ratio=0.8, val_ratio=0.1):
    n = len(data)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    return data[:train_end], data[train_end:val_end], data[val_end:]


def load_bio(filepath):
    sentences, tags = [], []
    cur_sent, cur_tags = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line == "":
                if cur_sent:
                    sentences.append(cur_sent)
                    tags.append(cur_tags)
                    cur_sent, cur_tags = [], []
            else:
                parts = line.split()
                if len(parts) == 2:
                    cur_sent.append(parts[0])
                    cur_tags.append(parts[1])
    if cur_sent:
        sentences.append(cur_sent)
        tags.append(cur_tags)
    return sentences, tags


def preprocess(sentences):
    processed = []
    for sent in sentences:
        new_sent = []
        for token in sent:
            token = token.lower().strip()
            new_sent.append(token if token else "<unk>")
        processed.append(new_sent)
    return processed


def build_vocab(sentences, tags):
    word_freq = defaultdict(int)
    tag_set   = set()
    char_set  = set()
    for sent, sent_tags in zip(sentences, tags):
        for token, tag in zip(sent, sent_tags):
            word_freq[token] += 1
            tag_set.add(tag)
            for ch in token:
                char_set.add(ch)
    word2idx = {"<pad>": 0, "<unk>": 1}
    for word in sorted(word_freq):
        word2idx[word] = len(word2idx)
    tag2idx = {"<pad>": 0}
    for tag in sorted(tag_set):
        tag2idx[tag] = len(tag2idx)
    char2idx = {"<pad>": 0, "<unk>": 1}
    for ch in sorted(char_set):
        char2idx[ch] = len(char2idx)
    return word2idx, tag2idx, char2idx


def encode(sentences, tags, word2idx, tag2idx, char2idx):
    X, y, X_char = [], [], []
    for sent, sent_tags in zip(sentences, tags):
        word_ids = [word2idx.get(t, word2idx["<unk>"]) for t in sent]
        tag_ids  = [tag2idx.get(t, 0) for t in sent_tags]
        char_ids = [
            [char2idx.get(ch, char2idx["<unk>"]) for ch in token]
            for token in sent
        ]
        X.append(word_ids)
        y.append(tag_ids)
        X_char.append(char_ids)
    return X, y, X_char


def pad(sequences, max_len, pad_value=0):
    padded = []
    for seq in sequences:
        seq = seq[:max_len]
        seq = seq + [pad_value] * (max_len - len(seq))
        padded.append(seq)
    return np.array(padded, dtype=np.int32)


def pad_chars(sequences, max_len, max_word_len, pad_value=0):
    padded = []
    for sent in sequences:
        sent = sent[:max_len]
        padded_sent = []
        for word in sent:
            word = word[:max_word_len]
            word = word + [pad_value] * (max_word_len - len(word))
            padded_sent.append(word)
        while len(padded_sent) < max_len:
            padded_sent.append([pad_value] * max_word_len)
        padded.append(padded_sent)
    return np.array(padded, dtype=np.int32)


def create_mask(X):
    return (X != 0).astype(np.int32)


# ─────────────────────────────────────────
# PIPELINE
# ─────────────────────────────────────────

# Step 1: Convert dataset to BIO format
data = []
for item in dataset:
    bio = convert_to_bio(item["text"], item["entities"])
    data.append(bio)

save_bio(data, "bio.txt")
print(f"✅ Converted {len(data)} documents to BIO format")

# Step 2: Split into train / val / test
train, val, test = split_data(data)
save_bio(train, "train.txt")
save_bio(val,   "val.txt")
save_bio(test,  "test.txt")
print(f"✅ Split data: Train={len(train)}, Val={len(val)}, Test={len(test)}")

# Step 3: Load train data
train_sents, train_tags = load_bio("train.txt")
print(f"✅ Loaded train: {len(train_sents)} sentences")

# Step 4: Preprocess
train_sents = preprocess(train_sents)
print(f"✅ Preprocessed tokens")

# Step 5: Build vocabulary
word2idx, tag2idx, char2idx = build_vocab(train_sents, train_tags)
print(f"✅ Built vocabulary: Words={len(word2idx)}, Tags={len(tag2idx)}, Chars={len(char2idx)}")

# Step 6: Encode
X, y, X_char = encode(train_sents, train_tags, word2idx, tag2idx, char2idx)
print(f"✅ Encoded data")

# Step 7: Padding
MAX_LEN      = max(len(s) for s in X)
MAX_WORD_LEN = max(len(w) for s in X_char for w in s)

X      = pad(X, MAX_LEN)
y      = pad(y, MAX_LEN)
X_char = pad_chars(X_char, MAX_LEN, MAX_WORD_LEN)
print(f"✅ Padded: Max Length={MAX_LEN}, Max Word Length={MAX_WORD_LEN}")

# Step 8: Create mask
mask = create_mask(X)
print(f"✅ Created attention mask")

print("\n" + "=" * 50)
print("✅ PIPELINE COMPLETED")
print("=" * 50)
print(f"Total sentences     : {len(X)}")
print(f"Max sequence length : {MAX_LEN}")
print(f"Word vocabulary size: {len(word2idx)}")
print(f"Tag vocabulary size : {len(tag2idx)}")
print(f"Char vocabulary size: {len(char2idx)}")

✅ Converted 200 documents to BIO format
✅ Split data: Train=160, Val=20, Test=20
✅ Loaded train: 160 sentences
✅ Preprocessed tokens
✅ Built vocabulary: Words=12377, Tags=83, Chars=88
✅ Encoded data
✅ Padded: Max Length=863, Max Word Length=82
✅ Created attention mask

✅ PIPELINE COMPLETED
Total sentences     : 160
Max sequence length : 863
Word vocabulary size: 12377
Tag vocabulary size : 83
Char vocabulary size: 88


In [14]:
# Shape Consistency Check
len(X) == len(y)

True

In [15]:
# Tag Integrity Check

idx2tag = {v: k for k, v in tag2idx.items()}

for seq in y:
    for i, tag_id in enumerate(seq):
        tag = idx2tag[tag_id]
        # Handle '<PAD>' tags which should not be checked for BIO integrity
        if tag == '<PAD>':
            continue

        if tag.startswith("I"):
            # If it's an 'I' tag at the beginning of a sequence or preceded by 'O'
            # this is an invalid BIO sequence.
            if i == 0 or idx2tag[seq[i-1]] == "O":
                print(f"Invalid BIO sequence found: 'I' tag '{tag}' at index {i} following '{idx2tag[seq[i-1]]}' (or at start of sequence).")


Invalid BIO sequence found: 'I' tag 'I-Dosage' at index 22 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Dosage' at index 34 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Lab_value' at index 353 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Disease_disorder' at index 80 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Disease_disorder' at index 250 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Detailed_description' at index 515 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Lab_value' at index 153 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Lab_value' at index 25 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-Diagnostic_procedure' at index 64 following 'O' (or at start of sequence).
Invalid BIO sequence found: 'I' tag 'I-

In [16]:
# Vocabulary Coverage

unknown_ratio = sum(1 for sent in X for w in sent if w == 1) / sum(len(s) for s in X)
print("UNK ratio:", unknown_ratio)

UNK ratio: 0.0


In [17]:
# Padding & Mask Check

print(mask[0])

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 

In [18]:
# Label Distribution

from collections import Counter
print(Counter(tag for seq in train_tags for tag in seq))

Counter({'O': 31164, 'B-Diagnostic_procedure': 3211, 'I-Diagnostic_procedure': 2592, 'B-Sign_symptom': 2560, 'B-Detailed_description': 2230, 'B-Biological_structure': 2229, 'B-Lab_value': 2051, 'I-Biological_structure': 1924, 'I-Lab_value': 1551, 'I-Detailed_description': 1506, 'I-Sign_symptom': 1105, 'I-Date': 1029, 'B-Disease_disorder': 1019, 'I-History': 821, 'B-Medication': 782, 'B-Therapeutic_procedure': 742, 'I-Disease_disorder': 655, 'B-Date': 547, 'B-Clinical_event': 479, 'I-Therapeutic_procedure': 424, 'I-Dosage': 383, 'I-Duration': 327, 'B-Severity': 303, 'B-History': 267, 'B-Coreference': 254, 'B-Nonbiological_location': 254, 'I-Nonbiological_location': 244, 'I-Family_history': 243, 'I-Medication': 222, 'B-Duration': 210, 'B-Dosage': 174, 'B-Age': 165, 'B-Sex': 154, 'B-Administration': 141, 'I-Distance': 136, 'I-Other_entity': 124, 'I-Area': 106, 'I-Coreference': 101, 'I-Time': 94, 'B-Distance': 90, 'B-Activity': 83, 'I-Frequency': 65, 'B-Frequency': 62, 'B-Family_history': 

Word Embedding Layer (PyTorch)

In [19]:
import torch
import torch.nn as nn

class WordEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

    def forward(self, x):
        return self.embedding(x)

In [20]:
#Usage

vocab_size = len(word2idx)
embed_dim = 100

word_embed = WordEmbedding(vocab_size, embed_dim)

sample = torch.tensor(X[:2])  # batch of sentences
out = word_embed(sample)

print(out.shape)
# (batch_size, seq_len, embed_dim)

torch.Size([2, 863, 100])


In [21]:
glove_path = r"C:\Users\sanan\OneDrive\Documents\NLP\NLP Capstone Project\GloVe\glove.6B.100d.txt"

Load Pretrained GloVe Embeddings

In [22]:
# Load GloVe File

import numpy as np
import torch

def load_glove_embeddings(glove_path):
    glove_dict = {}
    
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.strip().split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            glove_dict[word] = vector

    print(f"Loaded {len(glove_dict)} GloVe vectors")
    return glove_dict

In [23]:
# Create Embedding Matrix

def create_embedding_matrix(word2idx, glove_dict, embed_dim=100):
    vocab_size = len(word2idx)
    
    embedding_matrix = np.random.uniform(-0.25, 0.25, (vocab_size, embed_dim))

    found = 0
    for word, idx in word2idx.items():
        if word in glove_dict:
            embedding_matrix[idx] = glove_dict[word]
            found += 1

    print(f"Matched {found}/{vocab_size} words with GloVe")
    
    return torch.tensor(embedding_matrix, dtype=torch.float32)

In [24]:
# Create Word Embedding Layer

import torch.nn as nn

def build_word_embedding_layer(embedding_matrix):
    return nn.Embedding.from_pretrained(
        embedding_matrix,
        freeze=False,      # allow training
        padding_idx=0
    )

In [25]:
# Usage

glove_path = r"C:\Users\sanan\OneDrive\Documents\NLP\NLP Capstone Project\GloVe\glove.6B.100d.txt"

glove_dict = load_glove_embeddings(glove_path)
embedding_matrix = create_embedding_matrix(word2idx, glove_dict, embed_dim=100)

word_embedding = build_word_embedding_layer(embedding_matrix)

Loaded 400000 GloVe vectors
Matched 6036/12377 words with GloVe


Character Embedding Layer (BiLSTM)

In [26]:
class CharEmbedding(nn.Module):
    def __init__(self, char_vocab_size, char_embed_dim=30, hidden_dim=50):
        super(CharEmbedding, self).__init__()
        
        self.char_embed = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=char_embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        # x: (batch_size, seq_len, word_len)
        batch_size, seq_len, word_len = x.size()

        x = x.view(-1, word_len)  # (batch*seq_len, word_len)
        x = self.char_embed(x)    # (batch*seq_len, word_len, char_embed_dim)

        _, (h, _) = self.lstm(x)

        # Concatenate forward + backward
        h = torch.cat((h[0], h[1]), dim=1)

        return h.view(batch_size, seq_len, -1)

In [27]:
# Usage

char_model = CharEmbedding(len(char2idx))

sample_char = torch.tensor(X_char[:2])  # batch
char_out = char_model(sample_char)

print(char_out.shape)
# (batch_size, seq_len, 100)  → because bidirectional (50*2)

torch.Size([2, 863, 100])


Combine Word + Character Embeddings

In [28]:
class CombinedEmbedding(nn.Module):
    def __init__(self, word_embedding_layer, char_vocab_size):
        super(CombinedEmbedding, self).__init__()

        self.word_embed = word_embedding_layer
        self.char_embed = CharEmbedding(char_vocab_size)

    def forward(self, word_input, char_input):
        word_vec = self.word_embed(word_input)     # (B, L, 100)
        char_vec = self.char_embed(char_input)     # (B, L, 100)

        combined = torch.cat([word_vec, char_vec], dim=-1)

        return combined

In [29]:
# Usage

model_embed = CombinedEmbedding(word_embedding, len(char2idx))

word_tensor = torch.tensor(X[:2])
char_tensor = torch.tensor(X_char[:2])

output = model_embed(word_tensor, char_tensor)

print(output.shape)
# (batch_size, seq_len, 200)

torch.Size([2, 863, 200])


Sanity Check

In [30]:
print("Word Embedding Shape:", word_embedding(word_tensor).shape)
print("Char Embedding Shape:", char_model(char_tensor).shape)
print("Combined Shape:", output.shape)

Word Embedding Shape: torch.Size([2, 863, 100])
Char Embedding Shape: torch.Size([2, 863, 100])
Combined Shape: torch.Size([2, 863, 200])
